In [ ]:
# MS Lesion Segmentation Training (2 modalities: FLAIR + T2 by default)
#
# Theory:
# We train a 3D segmentation model that maps a multi-modal MRI volume x
# to a voxel-wise lesion probability map y_hat in [0,1]^(H x W x Z).
#
# Given severe class imbalance (few lesion voxels, many background voxels),
# the objective combines overlap-aware and pointwise terms.
#
# Goals:
# - Build a reproducible FLAIR-only baseline
# - Train with leave-one-out cross-validation (LOOCV)
# - Save fold checkpoints and metrics
# - Visualize precision-recall behavior and qualitative overlays

In [ ]:
# Math:
# Let p_i = sigmoid(z_i) be predicted probability, g_i in {0,1} the target.
#
# Dice loss:
# L_dice = 1 - (2 * sum_i p_i g_i + eps) / (sum_i p_i + sum_i g_i + eps)
#
# BCE loss:
# L_bce = -(1/N) * sum_i [g_i log(p_i) + (1-g_i) log(1-p_i)]
#
# Focal loss:
# L_focal = -(1/N) * sum_i [alpha*g_i*(1-p_i)^gamma*log(p_i)
#                           + (1-alpha)*(1-g_i)*p_i^gamma*log(1-p_i)]
#
# Combined loss:
# L = w_d * L_dice + w_b * L_bce + w_f * L_focal
#
# LOOCV trains on N-1 cases and validates on the held-out case for each fold.

In [ ]:
# Intuition:
# - Dice stabilizes overlap quality when positives are sparse.
# - BCE provides dense voxel-wise supervision.
# - Focal emphasizes hard/rare positives.
# - LOOCV uses data efficiently for small medical cohorts.
#
# Code sections:
# 1) Imports and deterministic setup
# 2) Config and path indexing
# 3) MONAI transforms and dataloaders
# 4) Model/loss/optimizer/scheduler factories
# 5) Training and validation engines
# 6) LOOCV runner with checkpointing
# 7) Aggregated metrics and Plotly visualizations

In [ ]:
# Imports and reproducibility
from __future__ import annotations

import copy
import json
import math
import os
import random
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import nibabel as nib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve, auc
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

from monai.data import CacheDataset, DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.networks.nets import UNet
try:
    from monai.transforms import ClipIntensityPercentilesd
except ImportError:
    ClipIntensityPercentilesd = None  # fallback to custom

from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    RandCropByPosNegLabeld,
    RandFlipd,
    SpatialPadd,
)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)
print('Imports complete. Seed fixed to 42.')

Imports complete. Seed fixed to 42.


In [ ]:
# Configuration
@dataclass
class TrainConfig:
    # Data
    data_root_rel: str = 'data/training'
    # Preprocessing: percentile clip (e.g. p0.5–p99.5) then mean-std (z-score) normalize.
    percentile_clip: Tuple[float, float] = (0, 100)  # (lower_pct, upper_pct) in 0–100
    # Modalities to concatenate (1–4). Options: flair, t2, mprage, pd.
    # Single modality: ('flair',) or 'flair'. Multi: ('flair', 't2') or ('flair', 't2', 'mprage').
    modalities: Tuple[str, ...] = ('flair', 't2', 'mprage')

    def __post_init__(self) -> None:
        if isinstance(self.modalities, str):
            self.modalities = (self.modalities,)
        else:
            self.modalities = tuple(self.modalities)
    cache_rate: float = 0.2
    num_workers: int = 0

    # Sampling / batching
    # Training: only use cases with lesions (masks with ones). Patch sampling pos/neg ratio.
    train_only_non_empty_masks: bool = True  # exclude blank masks from training
    pos_neg_ratio: Tuple[int, int] = (1, 1)  # (pos, neg) - favor lesion-containing patches
    batch_size: int = 2
    patch_size: Tuple[int, int, int] = (64, 96, 96)  # must be divisible by 8 for UNet
    num_samples_per_volume: int = 8  # more patches per epoch for better overlap learning

    # Optimization
    epochs: int = 125
    learning_rate: float = 2e-3
    weight_decay: float = 1e-5
    optimizer_name: str = 'adamw'  # adamw | adam
    scheduler_name: str = 'plateau'  # plateau | cosine | none
    grad_clip_norm: Optional[float] = 1

    # Loss: Dice directly optimizes IoU/overlap; Focal handles class imbalance.
    # Recommended: dice + focal for better IoU while loss decreases.
    loss_weights: Dict[str, float] = field(default_factory=lambda: {'dice': 0.2, 'bce': 0, 'focal': 0.3, 'tversky': 0.5})
    focal_alpha: float = 0.25  # 0.25 balances pos/neg when lesions are sparse
    focal_gamma: float = 1.5   # lower than 2.0 for more stable early training
    tversky_alpha: float = 0.2  # FP penalty: higher α penalizes FP more (0.3-0.7 typical)
    tversky_beta: float = 0.8   # FN penalty: higher β penalizes FN more (sum α+β=1 typical)

    # Inference / evaluation
    threshold: float = 0.5
    sw_batch_size: int = 1
    sw_overlap: float = 0.5
    eval_metrics: Tuple[str, ...] = ('dice', 'iou', 'tversky', 'precision', 'recall', 'pr_auc')

    # Model capacity and pretrained
    channels: Tuple[int, ...] = (32,64,128)  # smaller = faster; (16,32,64,128) for full
    num_res_units: int = 1  # 1 = faster than 2
    pretrained_path: Optional[str] = None  # outputs/.../fold_XX_best.pt, or MONAI Model Zoo
    freeze_encoder_stages: int = 0  # 0=none; 1=first stage; 2=first two; etc.

    # Runtime controls
    max_folds: Optional[int] = None  # set an int for quick smoke-runs
    save_outputs: bool = True


cfg = TrainConfig()
print(cfg)

TrainConfig(data_root_rel='data/training', percentile_clip=(0, 100), modalities=('flair', 't2', 'mprage'), cache_rate=0.2, num_workers=0, train_only_non_empty_masks=True, pos_neg_ratio=(1, 1), batch_size=2, patch_size=(64, 96, 96), num_samples_per_volume=8, epochs=125, learning_rate=0.002, weight_decay=1e-05, optimizer_name='adamw', scheduler_name='plateau', grad_clip_norm=1, loss_weights={'dice': 0.2, 'bce': 0, 'focal': 0.3, 'tversky': 0.5}, focal_alpha=0.25, focal_gamma=1.5, tversky_alpha=0.2, tversky_beta=0.8, threshold=0.5, sw_batch_size=1, sw_overlap=0.5, eval_metrics=('dice', 'iou', 'tversky', 'precision', 'recall', 'pr_auc'), channels=(32, 64, 128), num_res_units=1, pretrained_path=None, freeze_encoder_stages=0, max_folds=None, save_outputs=True)


In [ ]:
# Resolve project root and build modality-mask case index
# Available modalities: flair, t2, mprage, pd (file suffix: *_<mod>_pp.nii*)
MODALITY_SUFFIXES: Dict[str, str] = {
    'flair': '_flair_pp.nii',
    't2': '_t2_pp.nii',
    'mprage': '_mprage_pp.nii',
    'pd': '_pd_pp.nii',
}


def resolve_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd() if start is None else start
    for p in [start, *start.parents]:
        if (p / 'data' / 'training').exists():
            return p
    raise FileNotFoundError('Could not find project root containing data/training')


def case_id_from_modality_path(path: Path, modality: str) -> str:
    # Example: training01_01_flair_pp.nii -> training01_01
    suffix = MODALITY_SUFFIXES.get(modality, '_flair_pp.nii')
    return path.name.replace(suffix + '.gz', '').replace(suffix, '')


def _normalize_modalities(modalities: Any) -> Tuple[str, ...]:
    """Accept str or sequence; return tuple of 1–4 modality names."""
    if isinstance(modalities, str):
        return (modalities,)
    mods = tuple(modalities)
    if not 1 <= len(mods) <= 4:
        raise ValueError(f'modalities must have 1–4 elements. Got {mods}.')
    return mods


def build_modality_mask_index(
    project_root: Path,
    data_root_rel: str,
    modalities: Any,
    subdir_glob: str = 'training*',
) -> pd.DataFrame:
    """Build modality/mask pair index. Returns image_1, image_2, ..., image_N, label.
    Use subdir_glob='test*' for test data. modalities e.g. ('flair',), ('flair','t2'), or ('flair','t2','mprage')."""
    modalities = _normalize_modalities(modalities)
    for m in modalities:
        if m not in MODALITY_SUFFIXES:
            raise ValueError(f'Unknown modality "{m}". Choose from {list(MODALITY_SUFFIXES.keys())}.')
    data_root = project_root / data_root_rel
    rows: List[Dict[str, str]] = []

    for train_dir in sorted(data_root.glob(subdir_glob)):
        if not train_dir.is_dir():
            continue

        preproc_dir = train_dir / 'preprocessed'
        mask_dir = train_dir / 'masks'

        mod_paths: Dict[str, Dict[str, Path]] = {}
        for mod in modalities:
            pattern = f'*{MODALITY_SUFFIXES[mod]}*'
            for f in sorted(preproc_dir.glob(pattern)):
                cid = case_id_from_modality_path(f, mod)
                mod_paths.setdefault(cid, {})[mod] = f

        # Collect mask1 and mask2 separately, then OR them
        from collections import defaultdict
        mask_files: Dict[str, Dict[str, Path]] = defaultdict(dict)
        for m in mask_dir.glob('*.nii*'):
            if '_mask1' in m.name:
                stem = m.name.replace('_mask1.nii.gz', '').replace('_mask1.nii', '')
                mask_files[stem]['mask1'] = m
            elif '_mask2' in m.name:
                stem = m.name.replace('_mask2.nii.gz', '').replace('_mask2.nii', '')
                mask_files[stem]['mask2'] = m
            else:
                # single mask file (no mask1/mask2 suffix)
                stem = m.name.replace('.nii.gz', '').replace('.nii', '')
                mask_files[stem]['mask1'] = m
        
        mask_map: Dict[str, Dict[str, Path]] = dict(mask_files)

        for cid, paths in mod_paths.items():
            if cid not in mask_map or not all(mod in paths for mod in modalities):
                continue
            masks = mask_map[cid]
            row: Dict[str, str] = {'case_id': cid, 'patient_id': cid.split('_')[0]}
            if 'mask1' in masks:
                row['mask1'] = str(masks['mask1'])
            if 'mask2' in masks:
                row['mask2'] = str(masks['mask2'])
            for i, mod in enumerate(modalities):
                row[f'image_{i+1}'] = str(paths[mod])
            rows.append(row)

    if not rows:
        raise ValueError(f'No {"+".join(modalities)} mask pairs found. Check folder naming and paths.')
    df = pd.DataFrame(rows).drop_duplicates(subset=['case_id']).sort_values('case_id').reset_index(drop=True)
    if df.empty:
        raise ValueError(f'No {"+".join(modalities)} mask pairs found. Check folder naming and paths.')

    # Sanity-check shape alignment for all pairs.
    image_keys = [f'image_{i+1}' for i in range(len(modalities))]
    bad_pairs: List[str] = []
    for _, r in df.iterrows():
        mask_path = r.get('mask1') or r.get('mask2')
        if not mask_path:
            continue
        lbl_shape = nib.load(mask_path).shape
        for k in image_keys:
            if nib.load(r[k]).shape != lbl_shape:
                bad_pairs.append(f"{r['case_id']}: {k} shape != mask {lbl_shape}")
    if bad_pairs:
        raise ValueError('Found shape mismatches:\n' + '\n'.join(bad_pairs[:10]))

    return df


def filter_to_non_empty_masks(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows whose mask has at least one positive voxel (non-empty)."""
    keep: List[int] = []
    for i, row in df.iterrows():
        has_lesion = False
        if 'mask1' in row and row['mask1']:
            if np.any(nib.load(row['mask1']).get_fdata() > 0):
                has_lesion = True
        if not has_lesion and 'mask2' in row and row['mask2']:
            if np.any(nib.load(row['mask2']).get_fdata() > 0):
                has_lesion = True
        if has_lesion:
            keep.append(i)
    out = df.loc[keep].reset_index(drop=True)
    return out


project_root = resolve_project_root()
cases_df = build_modality_mask_index(
    project_root=project_root,
    data_root_rel=cfg.data_root_rel,
    modalities=cfg.modalities,
)

outputs_dir = project_root / 'outputs' / 'ms_seg' / f"{'_'.join(cfg.modalities)}_loocv"
if cfg.save_outputs:
    outputs_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Modalities: {cfg.modalities}')
print(f'Cases: {len(cases_df)}')
display(cases_df.head())

KeyError: 'label'

In [ ]:
cfg.modalities

In [ ]:
# Device + simple train (all data, no split)

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print(f'Using device: {device}')

# All data for training (no LOOCV, no val split)
train_df = cases_df
if cfg.train_only_non_empty_masks:
    train_df = filter_to_non_empty_masks(train_df)
    dropped = len(cases_df) - len(train_df)
    print(f'Filtered to {len(train_df)} cases with lesions (excluded {dropped} empty masks)')
    if train_df.empty:
        raise ValueError('No training cases with lesions. All masks are blank.')
val_df = train_df  # same data for validation metrics
splits = [(train_df, val_df)]  # single "fold" for compatibility
print(f'Training on {len(train_df)} cases.')

In [ ]:
def _make_or_masks():
    """Load mask1 and/or mask2, OR them together if both exist."""
    def _or(data: Dict[str, Any]) -> Dict[str, Any]:
        d = dict(data)
        mask1_path = d.get('mask1')
        mask2_path = d.get('mask2')
        
        if mask1_path and mask2_path:
            # Load both and OR
            mask1 = nib.load(mask1_path).get_fdata()
            mask2 = nib.load(mask2_path).get_fdata()
            combined = ((mask1 > 0) | (mask2 > 0)).astype(np.float32)
            d['label'] = combined
        elif mask1_path:
            d['label'] = nib.load(mask1_path).get_fdata().astype(np.float32)
        elif mask2_path:
            d['label'] = nib.load(mask2_path).get_fdata().astype(np.float32)
        else:
            raise ValueError(f"No mask found for case {d.get('case_id', 'unknown')}")
        
        # Clean up mask paths
        d.pop('mask1', None)
        d.pop('mask2', None)
        return d
    return _or


# Convert DataFrame rows to MONAI dictionaries

def to_data_dicts(df: pd.DataFrame, n_modalities: int) -> List[Dict[str, str]]:
    image_keys = [f'image_{i+1}' for i in range(n_modalities)]
    return [
        {**{k: getattr(r, k) for k in image_keys}, 'label': r.label, 'case_id': r.case_id}
        for r in df.itertuples(index=False)
    ]


def _make_percentile_clip(lower_pct: float, upper_pct: float):
    """Clip image intensities to [lower_pct, upper_pct] percentiles (0-100). Per-channel."""

    def _clip(data: Dict[str, Any]) -> Dict[str, Any]:
        d = dict(data)
        arr = np.asarray(d['image'])
        # Per-channel percentiles over spatial dims
        axes = tuple(range(1, arr.ndim))
        lo = np.percentile(arr, lower_pct, axis=axes, keepdims=True)
        hi = np.percentile(arr, upper_pct, axis=axes, keepdims=True)
        d['image'] = np.clip(arr, lo, hi).astype(arr.dtype)
        return d

    return _clip


def _make_concat_modalities(n: int):
    """Return a transform that stacks image_1..image_N along channel axis -> 'image'."""

    def _concat(data: Dict[str, Any]) -> Dict[str, Any]:
        d = dict(data)
        arrs = [np.asarray(d[f'image_{i+1}']) for i in range(n)]
        d['image'] = np.concatenate(arrs, axis=0)
        for i in range(n):
            del d[f'image_{i+1}']
        return d

    return _concat


def get_transforms(cfg: TrainConfig, train: bool) -> Compose:
    # UNet needs spatial dims divisible by 2^num_strides (8 for 3 strides)
    patch = tuple((s // 8) * 8 for s in cfg.patch_size)
    n = len(cfg.modalities)
    image_keys = [f'image_{i+1}' for i in range(n)]
    # Load image keys, but handle masks separately (OR mask1 + mask2)
    load_keys = image_keys

    common = [
        _make_or_masks(),  # Load and OR mask1/mask2 -> 'label'
        LoadImaged(keys=load_keys),
        EnsureChannelFirstd(keys=load_keys + ['label']),
        Orientationd(keys=load_keys + ['label'], axcodes='RAS'),
        _make_concat_modalities(n),
        (
            ClipIntensityPercentilesd(
                keys=['image'],
                lower=cfg.percentile_clip[0],
                upper=cfg.percentile_clip[1],
            )
            if ClipIntensityPercentilesd is not None
            else _make_percentile_clip(cfg.percentile_clip[0], cfg.percentile_clip[1])
        ),
        NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),  # mean-std (z-score)
        SpatialPadd(keys=['image', 'label'], spatial_size=patch),
        EnsureTyped(keys=['image', 'label']),
    ]
    if train:
        pos_w, neg_w = cfg.pos_neg_ratio
        aug = [
            RandCropByPosNegLabeld(
                keys=['image', 'label'],
                label_key='label',
                spatial_size=patch,
                pos=pos_w,
                neg=neg_w,
                num_samples=1,
            ),
            RandFlipd(keys=['image', 'label'], spatial_axis=0, prob=0.5),
            RandFlipd(keys=['image', 'label'], spatial_axis=1, prob=0.5),
            RandFlipd(keys=['image', 'label'], spatial_axis=2, prob=0.5),
        ]
        return Compose(common + aug)
    return Compose(common)

In [ ]:
# DataLoaders

def build_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, cfg: TrainConfig) -> Tuple[DataLoader, DataLoader]:
    n_mods = len(cfg.modalities)
    train_files = to_data_dicts(train_df, n_mods)
    val_files = to_data_dicts(val_df, n_mods)

    # Repeat each volume to get multiple random patches per epoch (num_samples=1 yields one dict per item)
    train_files_expanded = [f for f in train_files for _ in range(cfg.num_samples_per_volume)]

    train_ds = CacheDataset(
        data=train_files_expanded,
        transform=get_transforms(cfg, train=True),
        cache_rate=cfg.cache_rate,
        num_workers=cfg.num_workers,
    )
    val_ds = Dataset(data=val_files, transform=get_transforms(cfg, train=False))

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader


_train_loader, _val_loader = build_dataloaders(splits[0][0], splits[0][1], cfg)
first_batch = next(iter(_train_loader))
print('Train batch image:', tuple(first_batch['image'].shape), 'label:', tuple(first_batch['label'].shape))

In [ ]:
# Model factory (configurable channels, pretrained, encoder freeze)

def build_model(device: torch.device, cfg: TrainConfig) -> nn.Module:
    n = len(cfg.channels) - 1
    in_ch = len(cfg.modalities)  # 2 for flair+t2 concat
    model = UNet(
        spatial_dims=3,
        in_channels=in_ch,
        out_channels=1,
        channels=tuple(cfg.channels),
        strides=(2,) * n,
        num_res_units=cfg.num_res_units,
    ).to(device)

    # Load pretrained weights (e.g. from prior fold or MONAI model zoo)
    if cfg.pretrained_path:
        path = Path(cfg.pretrained_path)
        if path.exists():
            ckpt = torch.load(path, map_location=device)
            state = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
            missing, unexpected = model.load_state_dict(state, strict=False)
            print(f'Loaded pretrained from {path} (missing: {len(missing)}, unexpected: {len(unexpected)})')
        else:
            print(f'Pretrained path not found, training from scratch: {path}')

    # Freeze early down-sampling blocks (model.0=first encoder; model.1.0.0=second; etc.)
    # if cfg.freeze_encoder_stages > 0:
    #     prefixes = ['model.0.']  # first down block
    #     pfx = 'model.1.0.'
    #     for _ in range(cfg.freeze_encoder_stages - 1):
    #         prefixes.append(pfx + '0.')
    #         pfx = pfx + '1.0.'
    #     frozen = 0
    #     for name, p in model.named_parameters():
    #         if any(name.startswith(pre) for pre in prefixes):
    #             p.requires_grad = False
    #             frozen += p.numel()
    #     print(f'Froze {frozen:,} encoder parameters')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Model: {trainable:,} trainable / {total:,} total parameters')
    return model


model = build_model(device, cfg)

## Loss Function Theory

### Tversky Loss

**Mathematical Definition:**

$$
\text{TverskyLoss} = 1 - \frac{TP}{TP + \alpha \cdot FP + \beta \cdot FN}
$$

Where:
- $TP$ = True Positives
- $FP$ = False Positives  
- $FN$ = False Negatives
- $\alpha$ = penalty weight for FP
- $\beta$ = penalty weight for FN

**Intuition:**

Tversky loss is a **generalization of Dice loss** that allows asymmetric penalization of false positives and false negatives:

- **Dice loss**: Special case when $\alpha = \beta = 0.5$
- **Precision-focused**: $\alpha > \beta$ → penalizes FP more (fewer false alarms)
- **Recall-focused**: $\alpha < \beta$ → penalizes FN more (fewer missed lesions)

**Why $\alpha=0.2$, $\beta=0.8$ for MS lesion segmentation?**

In medical imaging, **missing a lesion (FN) is often more critical than a false alarm (FP)**:
- False Negative → Missed diagnosis, delayed treatment
- False Positive → Can be filtered by radiologist review

By setting $\beta > \alpha$, we optimize for **high recall** (sensitivity) while maintaining acceptable precision.

**Default Configuration:**
- `loss_weights`: `{'dice': 0.2, 'bce': 0, 'focal': 0.3, 'tversky': 0.5}` ← Tversky is primary
- `tversky_alpha = 0.2` (low FP penalty)
- `tversky_beta = 0.8` (high FN penalty)

---

In [ ]:
# Loss / optimizer / scheduler factories

class CombinedSegLoss(nn.Module):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.dice = DiceLoss(sigmoid=True)
        # Tversky: generalizes Dice; α=β=0.5 → Dice; α>β penalizes FP more (precision); α<β penalizes FN more (recall)
        from monai.losses import TverskyLoss
        self.tversky = TverskyLoss(sigmoid=True, alpha=cfg.tversky_alpha, beta=cfg.tversky_beta)

    def focal_from_logits(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        eps = 1e-7
        probs = torch.clamp(probs, eps, 1.0 - eps)
        alpha = self.cfg.focal_alpha
        gamma = self.cfg.focal_gamma

        pos_term = -alpha * ((1.0 - probs) ** gamma) * targets * torch.log(probs)
        neg_term = -(1.0 - alpha) * (probs**gamma) * (1.0 - targets) * torch.log(1.0 - probs)
        return (pos_term + neg_term).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        w = self.cfg.loss_weights
        loss = torch.tensor(0.0, device=logits.device)
        if w.get('dice', 0.0) > 0:
            loss = loss + w['dice'] * self.dice(logits, targets)
        if w.get('bce', 0.0) > 0:
            loss = loss + w['bce'] * F.binary_cross_entropy_with_logits(logits, targets)
        if w.get('focal', 0.0) > 0:
            loss = loss + w['focal'] * self.focal_from_logits(logits, targets)
        if w.get('tversky', 0.0) > 0:
            loss = loss + w['tversky'] * self.tversky(logits, targets)
        return loss


def build_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    if cfg.optimizer_name.lower() == 'adamw':
        return AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    if cfg.optimizer_name.lower() == 'adam':
        return Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    raise ValueError(f'Unsupported optimizer_name: {cfg.optimizer_name}')


def build_scheduler(optimizer: torch.optim.Optimizer, cfg: TrainConfig):
    name = cfg.scheduler_name.lower()
    if name == 'plateau':
        return ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    if name == 'cosine':
        return CosineAnnealingLR(optimizer, T_max=max(2, cfg.epochs))
    if name == 'none':
        return None
    raise ValueError(f'Unsupported scheduler_name: {cfg.scheduler_name}')


def maybe_load_pretrained(model: nn.Module, ckpt_path: Optional[Path]) -> None:
    if ckpt_path is None:
        return
    if ckpt_path.exists():
        state = torch.load(ckpt_path, map_location='cpu')
        model.load_state_dict(state['model'])
        print(f'Loaded checkpoint: {ckpt_path}')
    else:
        print(f'Checkpoint not found, training from scratch: {ckpt_path}')

In [ ]:
# Train/validation core steps

def batch_iou(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5) -> float:
    pred = (torch.sigmoid(logits) >= threshold).float()
    y = (targets > 0).float()
    tp = (pred * y).sum()
    fp = (pred * (1 - y)).sum()
    fn = ((1 - pred) * y).sum()
    eps = 1e-8
    return float((tp + eps) / (tp + fp + fn + eps))


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    cfg: TrainConfig,
) -> Tuple[float, float]:
    model.train()
    losses: List[float] = []
    ious: List[float] = []
    pbar = tqdm(loader, desc='train', unit='batch', leave=False)

    for batch in pbar:
        x = batch['image'].to(device)
        y = (batch['label'] > 0).float().to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)

        if not torch.isfinite(loss):
            raise ValueError('Encountered non-finite training loss.')

        loss.backward()
        if cfg.grad_clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        optimizer.step()

        loss_val = float(loss.detach().cpu())
        iou_val = batch_iou(logits.detach(), y, cfg.threshold)
        losses.append(loss_val)
        ious.append(iou_val)
        pbar.set_postfix(loss=f'{loss_val:.4f}', iou=f'{iou_val:.4f}')

    mean_loss = float(np.mean(losses)) if losses else math.nan
    mean_iou = float(np.mean(ious)) if ious else math.nan
    return mean_loss, mean_iou


def run_sliding_window_logits(model: nn.Module, x: torch.Tensor, cfg: TrainConfig) -> torch.Tensor:
    return sliding_window_inference(
        inputs=x,
        roi_size=cfg.patch_size,
        sw_batch_size=cfg.sw_batch_size,
        predictor=model,
        overlap=cfg.sw_overlap,
    )

In [ ]:
# Evaluation metrics (configurable: Dice, IoU, Tversky, Precision, Recall, PR-AUC)

def compute_overlap_metrics(
    probs: np.ndarray,
    targets: np.ndarray,
    threshold: float,
    alpha: float,
    beta: float,
) -> Dict[str, float]:
    p = (probs >= threshold).astype(np.float32)
    y = (targets > 0).astype(np.float32)

    tp = float((p * y).sum())
    fp = float((p * (1.0 - y)).sum())
    fn = float(((1.0 - p) * y).sum())
    eps = 1e-8

    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    flat_y = y.ravel()
    flat_probs = probs.ravel()
    pr_auc = math.nan
    if np.any(flat_y > 0):
        pr, rc, _ = precision_recall_curve(flat_y, flat_probs)
        pr_auc = float(auc(rc, pr))

    return {
        'dice': float(dice),
        'iou': float(iou),
        'tversky': float(tversky),
        'precision': float(precision),
        'recall': float(recall),
        'pr_auc': float(pr_auc),
    }


def validate_one_case(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    cfg: TrainConfig,
) -> Tuple[Dict[str, float], List[np.ndarray]]:
    model.eval()
    losses: List[float] = []
    probs_list: List[np.ndarray] = []
    labels_list: List[np.ndarray] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc='val', unit='case', leave=False):
            x = batch['image'].to(device)
            y = (batch['label'] > 0).float().to(device)

            logits = run_sliding_window_logits(model, x, cfg)
            loss = criterion(logits, y)
            loss_val = float(loss.detach().cpu())
            losses.append(loss_val)

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            labels = y.detach().cpu().numpy()
            probs_list.append(probs)
            labels_list.append(labels)

    probs_all = np.concatenate([p.ravel() for p in probs_list])
    labels_all = np.concatenate([l.ravel() for l in labels_list])

    metric_map = compute_overlap_metrics(
        probs=probs_all,
        targets=labels_all,
        threshold=cfg.threshold,
        alpha=cfg.tversky_alpha,
        beta=cfg.tversky_beta,
    )
    metric_map['val_loss'] = float(np.mean(losses)) if losses else math.nan

    return metric_map, [probs_list[0], labels_list[0]]


In [ ]:
# Single-fold runner (train only, no validation)

def run_one_fold(
    fold_idx: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    cfg: TrainConfig,
    device: torch.device,
    outputs_dir: Path,
) -> Tuple[Dict[str, Any], pd.DataFrame, Dict[str, Any]]:
    train_loader, _ = build_dataloaders(train_df, val_df, cfg)

    model = build_model(device, cfg)
    criterion = CombinedSegLoss(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    fold_history: List[Dict[str, float]] = []
    best_iou = -1.0
    best_train_loss = float('inf')
    best_ckpt = outputs_dir / f'fold_{fold_idx:02d}_best.pt'

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_iou = train_one_epoch(model, train_loader, criterion, optimizer, device, cfg)

        row = {'fold': fold_idx, 'epoch': epoch, 'train_loss': train_loss, 'iou': train_iou}
        fold_history.append(row)

        if train_iou > best_iou:
            best_iou = train_iou
            best_train_loss = train_loss
            if cfg.save_outputs:
                torch.save(
                    {
                        'fold': fold_idx,
                        'epoch': epoch,
                        'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'config': asdict(cfg),
                    },
                    best_ckpt,
                )

        if scheduler is not None:
            if isinstance(scheduler, ReduceLROnPlateau):
                scheduler.step(train_loss)
            else:
                scheduler.step()

        print(f"Epoch {epoch:03d}/{cfg.epochs} | train_loss={train_loss:.4f} | iou={train_iou:.4f} | best_iou={best_iou:.4f}")

    history_df = pd.DataFrame(fold_history)
    fold_summary: Dict[str, Any] = {
        'fold': fold_idx,
        'best_iou': best_iou,
        'best_train_loss': best_train_loss,
        'checkpoint_path': str(best_ckpt) if cfg.save_outputs else '',
    }

    qualitative = {'fold': fold_idx, 'case_id': '', 'probs': None, 'labels': None, 'metrics': {}}
    return fold_summary, history_df, qualitative

In [ ]:
# Single training run (all data)
fold_summaries: List[Dict[str, Any]] = []
fold_histories: List[pd.DataFrame] = []
qualitative_samples: List[Dict[str, Any]] = []

for fold_idx, (train_df, val_df) in enumerate(splits, start=1):
    print(f"\n=== Training run | {len(train_df)} cases ===")
    summary, hist_df, qual = run_one_fold(
        fold_idx=fold_idx,
        train_df=train_df,
        val_df=val_df,
        cfg=cfg,
        device=device,
        outputs_dir=outputs_dir,
    )
    fold_summaries.append(summary)
    fold_histories.append(hist_df)
    qualitative_samples.append(qual)

print('\nTraining complete.')


In [ ]:
# Persist fold metrics and training history
fold_summary_df = pd.DataFrame(fold_summaries).sort_values('fold').reset_index(drop=True)
all_history_df = pd.concat(fold_histories, ignore_index=True) if fold_histories else pd.DataFrame()

if cfg.save_outputs:
    fold_summary_path = outputs_dir / 'fold_summary.csv'
    history_path = outputs_dir / 'train_history.csv'
    config_path = outputs_dir / 'train_config.json'

    fold_summary_df.to_csv(fold_summary_path, index=False)
    all_history_df.to_csv(history_path, index=False)
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(asdict(cfg), f, indent=2)

    print('Saved:')
    print('-', fold_summary_path)
    print('-', history_path)
    print('-', config_path)

display(fold_summary_df)
if not all_history_df.empty:
    display(all_history_df.head())

In [ ]:
# Plot training loss over epochs
if all_history_df.empty:
    raise ValueError('History is empty. Run training cell first.')

fig_loss = px.line(
    all_history_df,
    x='epoch',
    y='train_loss',
    color='fold',
    markers=True,
    template='plotly_white',
    title='Training Loss by Epoch',
)
fig_loss.update_layout(height=420, width=900)
fig_loss.show()

In [ ]:
# Precision-Recall curves (skipped when train-only, no validation samples)
valid_samples = [s for s in qualitative_samples if s.get('probs') is not None and s.get('labels') is not None]
if not valid_samples:
    print('No validation samples for PR curve (train-only mode).')
else:
    def pr_from_qual_sample(sample):
        y = sample['labels'].ravel().astype(np.int32)
        p = sample['probs'].ravel().astype(np.float32)
        pr, rc, _ = precision_recall_curve(y, p)
        pr_auc = float(auc(rc, pr)) if np.any(y > 0) else math.nan
        return rc, pr, pr_auc

    fig_pr = go.Figure()
    for sample in valid_samples:
        rc, pr, pr_auc = pr_from_qual_sample(sample)
        fig_pr.add_trace(go.Scatter(x=rc, y=pr, mode='lines', name=f"Fold {sample['fold']} AUC={pr_auc:.3f}"))
    fig_pr.update_layout(template='plotly_white', title='Precision-Recall', xaxis_title='Recall', yaxis_title='Precision', width=1100, height=600)
    fig_pr.show()

In [ ]:
# Qualitative overlay for one LOOCV sample

def show_overlay(sample: Dict[str, Any], z_index: Optional[int] = None) -> None:
    probs = sample['probs'][0, 0]  # (H, W, Z)
    label = sample['labels'][0, 0]
    pred = (probs >= cfg.threshold).astype(np.float32)

    z_index = probs.shape[-1] // 2 if z_index is None else int(z_index)

    fig = make_subplots(rows=1, cols=3, subplot_titles=['Ground truth', 'Prediction', 'Overlay'])
    gt_slice = label[:, :, z_index].T
    pr_slice = pred[:, :, z_index].T
    ov_slice = gt_slice + 2.0 * pr_slice

    fig.add_trace(go.Heatmap(z=gt_slice, colorscale='Reds', showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=pr_slice, colorscale='Blues', showscale=False), row=1, col=2)
    fig.add_trace(go.Heatmap(z=ov_slice, colorscale='Viridis', showscale=False), row=1, col=3)

    for c in [1, 2, 3]:
        fig.update_xaxes(showticklabels=False, row=1, col=c)
        fig.update_yaxes(showticklabels=False, row=1, col=c)

    fig.update_layout(
        template='plotly_white',
        width=1150,
        height=380,
        title_text=f"Qualitative check | Fold {sample['fold']} | Case {sample['case_id']} | Z={z_index}",
    )
    fig.show()


valid_overlay = [s for s in qualitative_samples if s.get('probs') is not None and s.get('labels') is not None]
if valid_overlay:
    show_overlay(valid_overlay[0])
else:
    print('No qualitative samples (train-only mode).')

In [ ]:
# Aggregate summary table (mean +- std)
if fold_summary_df.empty:
    raise ValueError('No summary available. Run training first.')

summary_cols = [c for c in ['best_train_loss', 'train_loss'] if c in fold_summary_df.columns]
if not summary_cols:
    summary_cols = [c for c in fold_summary_df.columns if fold_summary_df[c].dtype in (np.float64, np.int64)]
summary_stats = pd.DataFrame(
    {
        'metric': summary_cols,
        'mean': [float(fold_summary_df[c].mean()) for c in summary_cols],
        'std': [float(fold_summary_df[c].std(ddof=0)) for c in summary_cols],
    }
)
summary_stats['mean_plus_minus_std'] = summary_stats.apply(lambda r: f"{r['mean']:.4f} +- {r['std']:.4f}", axis=1)

if cfg.save_outputs:
    stats_path = outputs_dir / 'summary_stats.csv'
    summary_stats.to_csv(stats_path, index=False)
    print(f'Saved: {stats_path}')

display(fold_summary_df)
display(summary_stats)

## Test Set Prediction and Evaluation

Load the best checkpoint, run **full-volume** sliding-window inference (covering all dimensions, including blank regions, for real-time deployment). Then compute Dice, IoU, Tversky, precision, recall, and PR-AUC.

In [ ]:
# Build test index: prefer data/test (test* subdirs); fallback to holdout from training
test_data_root = project_root / 'data' / 'test'
test_df: pd.DataFrame

try:
    test_df = build_modality_mask_index(
        project_root=project_root,
        data_root_rel='data/test',
        modalities=cfg.modalities,
        subdir_glob='test*',
    )
    print(f'Test set: {len(test_df)} cases from data/test')
except (ValueError, FileNotFoundError):
    # No data/test: use last 20% of training as holdout (min 1, max 5)
    n_holdout = max(1, min(5, len(cases_df) // 5))
    test_df = cases_df.tail(n_holdout).reset_index(drop=True)
    print(f'[Fallback] No data/test found. Using last {n_holdout} training cases as holdout.')
    print('To use a dedicated test set, add data/test/ with test01/, test02/, ... (preprocessed/, masks/)')

In [ ]:
# Load best checkpoint and run prediction + evaluation on test set
best_ckpt = outputs_dir / 'fold_01_best.pt'
if not best_ckpt.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_ckpt}. Run training first.')

test_model = build_model(device, cfg)
ckpt = torch.load(best_ckpt, map_location=device)
state = ckpt.get('model', ckpt)
test_model.load_state_dict(state, strict=True)
test_model.eval()

test_files = to_data_dicts(test_df, len(cfg.modalities))
test_ds = Dataset(data=test_files, transform=get_transforms(cfg, train=False))
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

# Run sliding-window inference per case and collect metrics
test_metrics_rows: List[Dict[str, Any]] = []
all_probs, all_labels = [], []
test_overlay_sample: Optional[Dict[str, Any]] = None  # first case for qualitative viz

with torch.no_grad():
    for batch in tqdm(test_loader, desc='test inference', unit='case'):
        x = batch['image'].to(device)
        y = (batch['label'] > 0).float()
        case_id = batch['case_id'][0] if isinstance(batch['case_id'], (list, tuple)) else str(batch['case_id'])

        logits = run_sliding_window_logits(test_model, x, cfg)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels = y.detach().cpu().numpy()

        all_probs.append(probs.ravel())
        all_labels.append(labels.ravel())

        m = compute_overlap_metrics(
            probs=probs.ravel(),
            targets=labels.ravel(),
            threshold=cfg.threshold,
            alpha=cfg.tversky_alpha,
            beta=cfg.tversky_beta,
        )
        test_metrics_rows.append({'case_id': case_id, **m})

        if test_overlay_sample is None:
            test_overlay_sample = {'probs': probs, 'labels': labels, 'case_id': case_id, 'fold': 'test'}

test_metrics_df = pd.DataFrame(test_metrics_rows)
test_all_probs = np.concatenate(all_probs)
test_all_labels = np.concatenate(all_labels)
aggregate = compute_overlap_metrics(
    probs=test_all_probs,
    targets=test_all_labels,
    threshold=cfg.threshold,
    alpha=cfg.tversky_alpha,
    beta=cfg.tversky_beta,
)

In [ ]:
# Test set evaluation summary
print('=== Test Set Evaluation ===')
print(f'Aggregate ({len(test_metrics_df)} cases):')
for k, v in aggregate.items():
    print(f'  {k}: {v:.4f}')

print('\nPer-case metrics:')
display(test_metrics_df)

if cfg.save_outputs:
    out_path = outputs_dir / 'test_metrics.csv'
    test_metrics_df.to_csv(out_path, index=False)
    print(f'\nSaved: {out_path}')

In [ ]:
model

In [ ]:
# Precision-Recall curve on test set
if np.any(test_all_labels > 0):
    precision_curve, recall_curve, _ = precision_recall_curve(
        test_all_labels.astype(np.int32), test_all_probs.astype(np.float32)
    )
    pr_auc_val = float(auc(recall_curve, precision_curve))

    fig_pr = go.Figure()
    fig_pr.add_trace(go.Scatter(
        x=recall_curve, y=precision_curve, mode='lines',
        name=f'Test PR (AUC={pr_auc_val:.3f})', line=dict(width=2)
    ))
    # Baseline: no-skill classifier (horizontal at prevalence)
    prevalence = float(test_all_labels.sum()) / max(1, test_all_labels.size)
    fig_pr.add_hline(y=prevalence, line_dash='dash', line_color='gray', annotation_text='No-skill')
    fig_pr.update_layout(
        template='plotly_white',
        title='Test Set: Precision-Recall Curve',
        xaxis_title='Recall',
        yaxis_title='Precision',
        width=800,
        height=550,
        showlegend=True,
    )
    fig_pr.show()
else:
    print('No positive labels in test set; cannot plot Precision-Recall curve.')

In [ ]:
# Qualitative overlay for first test case
if test_overlay_sample is not None:
    show_overlay(test_overlay_sample)
else:
    print('No test samples for overlay.')

In [ ]:
# How to run:
# 1) Execute cells top-to-bottom.
# 2) Quick smoke test: cfg.max_folds=2 and cfg.epochs=2.
# 3) Full run: cfg.max_folds=None.
#
# Artifacts under outputs/ms_seg/flair_loocv/:
# - fold_XX_best.pt
# - fold_summary.csv
# - train_history.csv
# - summary_stats.csv
# - train_config.json
#
# Notes:
# - Baseline is FLAIR-only.
# - Loss is parameterized (dice/bce/focal).
# - Optimizer is parameterized (adamw/adam).
# - Metrics include dice, iou, tversky, precision, recall, pr_auc.